In [0]:
%python
dbutils.widgets.dropdown(name = 'Environment', defaultValue = 'dev', choices = ['qa','dev','prd'], label = 'Environment')
env = dbutils.widgets.get("Environment")

In [0]:
%python
silverTableName = f"saleslake_{env}.silver_{env}.cleanedCustomer"
print(silverTableName)

bronzeTableName = f"saleslake_{env}.bronze_{env}.rawcustomer"
print(bronzeTableName)

srcFileLoc=f"/Volumes/saleslake_{env}/silver_{env}/vol_saleslake_src_files_{env}/daily_customer/"
print(srcFileLoc)


In [0]:
spark.sql(f"""INSERT INTO {silverTableName}
SELECT DISTINCT  
    CAST(TRIM(customer_id) AS INTEGER) as customer_id ,
    UPPER(TRIM(customer_name)) as customer_name,
    UPPER(TRIM(email)) as email,
    UPPER(TRIM(phone)) as phone,
    UPPER(TRIM(address)) as address,
    UPPER(TRIM(city)) as city,
    UPPER(TRIM(state)) as state,
    UPPER(TRIM(country)) as country,
    UPPER(TRIM(zip_code)) as zip_code,
    UPPER(TRIM(segment)) as segment,
    CURRENT_TIMESTAMP() as ingest_ts
FROM {bronzeTableName}
WHERE ingest_ts > (
                    SELECT coalesce(MAX(ingest_ts),TO_DATE('1990-01-01','yyyy-MM-dd')) 
                    FROM {silverTableName}
                    )
ORDER BY CAST(TRIM(customer_id) AS INTEGER)
""")

In [0]:
# %python
# from pyspark.sql.functions import coalesce, max as spark_max, col, lit, upper, trim, current_timestamp
# from pyspark.sql.types import IntegerType

# ingest_ts = df_silver.select(coalesce(
#     spark_max("ingest_ts"),
#     lit("1900-01-01").cast("timestamp")
# ).alias("max_ts")).collect()[0]["max_ts"]

# df_cleaned = (
#     df_raw
#     .filter(col("ingest_ts") > lit(ingest_ts))
#     .select(
#         col("customer_id").cast(IntegerType()).alias("customer_id"),
#         upper(trim(col("customer_name"))).alias("customer_name"),
#         upper(trim(col("email"))).alias("email"),
#         upper(trim(col("phone"))).alias("phone"),
#         upper(trim(col("address"))).alias("address"),
#         upper(trim(col("city"))).alias("city"),
#         upper(trim(col("state"))).alias("state"),
#         upper(trim(col("country"))).alias("country"),
#         col("zip_code").cast(IntegerType()).alias("zip_code"),
#         upper(trim(col("segment"))).alias("segment"),
#         current_timestamp().alias("ingest_ts")
#     )
#     .distinct()
#     .orderBy(col("customer_id"))
# )

# # Insert into silver table
# df_cleaned.write.mode("append").insertInto(
#     "intellibi_catalog.intellibi_silver.cleanedCustomer"
# )

In [0]:
%python
#display(df_cleaned)

In [0]:
%sql
--SELECT * FROM saleslake_dev.silver_dev.cleanedcustomer;